# MCTS Demo

Step-by-step walkthrough of Monte Carlo Tree Search on Bridgit with the 1-2-2 turn structure.

## 1. Setup — create net, wrapper, MCTS

In [1]:
import numpy as np
from bridgit import Bridgit, Player
from bridgit.ai import BridgitNet, NetWrapper, MCTS, Config, GameWrapper

N = 5  # small board for fast demo
config = Config(board_size=N, num_mcts_sims=5000)
net = BridgitNet(board_size=N, num_channels=32, num_res_blocks=2)
wrapper = NetWrapper(net)
mcts = MCTS(wrapper, config)
gw = GameWrapper(N)

print(f"Board: {config.grid_size}x{config.grid_size}")
print(f"Action space: {config.action_size} (flat), {len(Bridgit(N).get_available_moves())} legal crossings")
print(f"Device: {wrapper.device}")

Board: 11x11
Action space: 121 (flat), 41 legal crossings
Device: mps


## 2. Raw neural net output (before any search)

The untrained net gives random-ish policy and value. Let's see what it produces on an empty board.

In [2]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

game = Bridgit(N)
g = config.grid_size

# Get raw net prediction
policy, value = mcts._predict(game)
policy_2d = policy.reshape(g, g)

# Mask to only legal crossings
mask = gw.get_valid_moves_mask(game).reshape(g, g)
masked_policy = policy_2d * mask
masked_policy /= masked_policy.sum()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Raw Policy (all cells)", "Masked Policy (legal moves only)"])

fig.add_trace(go.Heatmap(z=np.flipud(policy_2d), colorscale="Blues", showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=np.flipud(masked_policy), colorscale="Reds", showscale=False), row=1, col=2)

fig.update_layout(width=700, height=350, title=f"Net value estimate: {value:.3f}")
fig.show()

## 3. Single MCTS search — visit counts and action probs

Run 50 simulations on the empty board. Compare raw policy (net alone) vs MCTS policy (net + search).

In [3]:
game = Bridgit(N)

# MCTS search
visit_counts = mcts.search(game)
mcts_probs = mcts.get_action_probs(game, temperature=1.0)

visits_2d = visit_counts.reshape(g, g)
mcts_2d = mcts_probs.reshape(g, g)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Visit Counts", "MCTS Policy (temp=1)"])

fig.add_trace(go.Heatmap(z=np.flipud(visits_2d), colorscale="Greens", showscale=True), row=1, col=1)
fig.add_trace(go.Heatmap(z=np.flipud(mcts_2d), colorscale="Reds", showscale=True), row=1, col=2)

fig.update_layout(width=700, height=350, title="MCTS concentrates visits on promising moves")
fig.show()

# Print top moves
print(f"Player: {game.current_player.name}, moves_left: {game.moves_left_in_turn}")
print(f"\nTop 5 moves by visit count:")
legal_actions = np.nonzero(gw.get_valid_moves_mask(game))[0]
sorted_actions = sorted(legal_actions, key=lambda a: visit_counts[a], reverse=True)
for a in sorted_actions[:5]:
    r, c = gw.action_to_move(a)
    print(f"  ({r},{c})  visits={int(visit_counts[a]):3d}  prob={mcts_probs[a]:.3f}")

Player: HORIZONTAL, moves_left: 1

Top 5 moves by visit count:
  (5,3)  visits=216  prob=0.022
  (2,2)  visits=211  prob=0.023
  (7,7)  visits=209  prob=0.023
  (5,1)  visits=205  prob=0.022
  (3,9)  visits=121  prob=0.022


## 4. Verify 1-2-2 turn structure in the tree

Inspect the root and first two levels. Root is P1 with 1 move → children are P2 with 2 moves → grandchildren are P2 with 1 move (same player!).

In [4]:
from bridgit.ai.mcts import MCTSNode

# Build a fresh tree so we can inspect it
game = Bridgit(N)
root = MCTSNode(game.copy())
mcts._expand(root)

# Run a few sims to populate the tree
for _ in range(config.num_mcts_sims):
    node = root
    while node.is_expanded and node.children:
        node = node.best_child(config.c_puct)
    if node.game.game_over:
        mcts._backpropagate(node, 1.0)
        continue
    value = mcts._expand(node)
    mcts._backpropagate(node, value)

# Inspect tree structure
print(f"ROOT: player={root.game.current_player.name}, moves_left={root.game.moves_left_in_turn}")
print(f"  children: {len(root.children)}\n")

# Pick the most visited child
best_action = max(root.children, key=lambda a: root.children[a].visit_count)
child = root.children[best_action]
r, c = gw.action_to_move(best_action)
print(f"CHILD (move {r},{c}): player={child.game.current_player.name}, moves_left={child.game.moves_left_in_turn}")
print(f"  visits={child.visit_count}, Q={child.q_value:.3f}")
print(f"  children: {len(child.children)}")

if child.children:
    # Pick most visited grandchild
    best_gc = max(child.children, key=lambda a: child.children[a].visit_count)
    gc = child.children[best_gc]
    r2, c2 = gw.action_to_move(best_gc)
    print(f"\nGRANDCHILD (move {r2},{c2}): player={gc.game.current_player.name}, moves_left={gc.game.moves_left_in_turn}")
    print(f"  visits={gc.visit_count}, Q={gc.q_value:.3f}")
    print(f"  children: {len(gc.children)}")

    # Verify: child and grandchild should be the SAME player (2-move turn)
    same = child.game.current_player == gc.game.current_player
    print(f"\n>>> Child and grandchild same player? {same} (expected: True for 2-move turn)")

ROOT: player=HORIZONTAL, moves_left=1
  children: 41

CHILD (move 3,7): player=VERTICAL, moves_left=2
  visits=209, Q=-0.017
  children: 40

GRANDCHILD (move 2,2): player=VERTICAL, moves_left=1
  visits=6, Q=-0.027
  children: 39

>>> Child and grandchild same player? True (expected: True for 2-move turn)


## 5. Full game — MCTS vs MCTS

Play a complete game, visualizing each board state. Each player uses the same untrained net.

In [5]:
game = Bridgit(N)
history = []

while not game.game_over:
    player = game.current_player
    left = game.moves_left_in_turn
    probs = mcts.get_action_probs(game, temperature=0.5)
    action = np.random.choice(len(probs), p=probs)
    row, col = gw.action_to_move(action)
    game.make_move(row, col)
    history.append((row, col, player.name, left))
    print(f"Move {len(history):2d}: ({row},{col}) by {player.name:10s}  (had {left} left in turn)")

print(f"\nWinner: {game.winner.name} in {len(history)} moves")

Move  1: (9,9) by HORIZONTAL  (had 1 left in turn)
Move  2: (8,8) by VERTICAL    (had 2 left in turn)
Move  3: (2,6) by VERTICAL    (had 1 left in turn)
Move  4: (7,5) by HORIZONTAL  (had 2 left in turn)
Move  5: (1,7) by HORIZONTAL  (had 1 left in turn)
Move  6: (7,3) by VERTICAL    (had 2 left in turn)
Move  7: (6,8) by VERTICAL    (had 1 left in turn)
Move  8: (5,5) by HORIZONTAL  (had 2 left in turn)
Move  9: (4,8) by HORIZONTAL  (had 1 left in turn)
Move 10: (1,3) by VERTICAL    (had 2 left in turn)
Move 11: (9,3) by VERTICAL    (had 1 left in turn)
Move 12: (5,3) by HORIZONTAL  (had 2 left in turn)
Move 13: (3,5) by HORIZONTAL  (had 1 left in turn)
Move 14: (3,3) by VERTICAL    (had 2 left in turn)
Move 15: (7,1) by VERTICAL    (had 1 left in turn)
Move 16: (2,8) by HORIZONTAL  (had 2 left in turn)
Move 17: (9,5) by HORIZONTAL  (had 1 left in turn)
Move 18: (2,4) by VERTICAL    (had 2 left in turn)
Move 19: (3,9) by VERTICAL    (had 1 left in turn)
Move 20: (6,6) by HORIZONTAL  (

## 6. Interactive Replay — visualize each move with slider

Use the slider to navigate through every move of the game. The play button (▶) automatically steps through the moves.

In [ ]:
# Interactive replay with slider — visualize board state after each move
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Build all game states
replay_states = []
replay = Bridgit(N)
replay_states.append(replay.copy())  # Initial empty board

for row, col, player_name, moves_left in history:
    replay.make_move(row, col)
    replay_states.append(replay.copy())

# Create frames for each state
g = 2 * N + 1
frames = []

for frame_idx, game in enumerate(replay_states):
    board = game.state.board
    frame_data = []
    
    # 1. Boundary dots (green and red goals)
    green_x, green_y = [], []
    red_x, red_y = [], []
    for c in range(1, g - 1, 2):
        green_x.extend([c, c])
        green_y.extend([0, g - 1])
    for r in range(1, g - 1, 2):
        red_x.extend([0, g - 1])
        red_y.extend([r, r])
    
    frame_data.append(go.Scatter(
        x=green_x, y=green_y, mode="markers",
        marker=dict(size=8, color="green", opacity=0.3),
        showlegend=False, hoverinfo="skip"
    ))
    frame_data.append(go.Scatter(
        x=red_x, y=red_y, mode="markers",
        marker=dict(size=8, color="red", opacity=0.3),
        showlegend=False, hoverinfo="skip"
    ))
    
    # 2. Ghost bridges (potential moves) as line segments
    ghost_green_x, ghost_green_y = [], []
    ghost_red_x, ghost_red_y = [], []
    
    for r in range(1, g - 1):
        for c in range(1, g - 1):
            if (r + c) % 2 != 0 or board[r, c] != 0:
                continue
            for player, color in [(Player.VERTICAL, "green"), (Player.HORIZONTAL, "red")]:
                eps = game.state.endpoints(r, c, player)
                (r0, c0), (r1, c1) = eps
                if color == "green":
                    ghost_green_x.extend([c0, c1, None])
                    ghost_green_y.extend([r0, r1, None])
                else:
                    ghost_red_x.extend([c0, c1, None])
                    ghost_red_y.extend([r0, r1, None])
    
    frame_data.append(go.Scatter(
        x=ghost_green_x, y=ghost_green_y, mode="lines",
        line=dict(color="green", width=2), opacity=0.2,
        showlegend=False, hoverinfo="skip"
    ))
    frame_data.append(go.Scatter(
        x=ghost_red_x, y=ghost_red_y, mode="lines",
        line=dict(color="red", width=2), opacity=0.2,
        showlegend=False, hoverinfo="skip"
    ))
    
    # 3. Actual bridges and endpoints
    bridge_green_x, bridge_green_y = [], []
    bridge_red_x, bridge_red_y = [], []
    endpoint_green_x, endpoint_green_y = [], []
    endpoint_red_x, endpoint_red_y = [], []
    
    for r in range(1, g - 1):
        for c in range(1, g - 1):
            if (r + c) % 2 != 0:
                continue
            val = board[r, c]
            if val == 0:
                continue
            player = Player.VERTICAL if val == 1 else Player.HORIZONTAL
            color = "green" if val == 1 else "red"
            eps = game.state.endpoints(r, c, player)
            (r0, c0), (r1, c1) = eps
            
            if color == "green":
                bridge_green_x.extend([c0, c1, None])
                bridge_green_y.extend([r0, r1, None])
                endpoint_green_x.extend([c0, c1])
                endpoint_green_y.extend([r0, r1])
            else:
                bridge_red_x.extend([c0, c1, None])
                bridge_red_y.extend([r0, r1, None])
                endpoint_red_x.extend([c0, c1])
                endpoint_red_y.extend([r0, r1])
    
    frame_data.append(go.Scatter(
        x=bridge_green_x, y=bridge_green_y, mode="lines",
        line=dict(color="green", width=5),
        showlegend=False, hoverinfo="skip"
    ))
    frame_data.append(go.Scatter(
        x=bridge_red_x, y=bridge_red_y, mode="lines",
        line=dict(color="red", width=5),
        showlegend=False, hoverinfo="skip"
    ))
    frame_data.append(go.Scatter(
        x=endpoint_green_x, y=endpoint_green_y, mode="markers",
        marker=dict(size=8, color="green"),
        showlegend=False, hoverinfo="skip"
    ))
    frame_data.append(go.Scatter(
        x=endpoint_red_x, y=endpoint_red_y, mode="markers",
        marker=dict(size=8, color="red"),
        showlegend=False, hoverinfo="skip"
    ))
    
    # Frame name
    if frame_idx == 0:
        frame_name = "Initial"
    else:
        row, col, player_name, moves_left = history[frame_idx - 1]
        frame_name = f"{frame_idx}: ({row},{col}) {player_name}"
    
    frames.append(go.Frame(data=frame_data, name=str(frame_idx)))

# Create figure with initial frame
fig = go.Figure(data=frames[0].data, frames=frames)

# Create slider steps
steps = []
for i in range(len(frames)):
    if i == 0:
        label = "Initial"
    else:
        row, col, player_name, moves_left = history[i - 1]
        label = f"{i}: ({row},{col}) {player_name[:4]}"
    
    step = dict(
        method="animate",
        args=[[str(i)], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))],
        label=label
    )
    steps.append(step)

# Add slider and play/pause buttons
sliders = [dict(
    active=0,
    yanchor="top",
    y=-0.15,
    xanchor="left",
    currentvalue=dict(prefix="Move: ", visible=True, xanchor="left"),
    pad=dict(b=10, t=50),
    len=0.9,
    x=0.05,
    steps=steps
)]

fig.update_layout(
    sliders=sliders,
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        y=-0.25,
        x=0.05,
        xanchor="left",
        yanchor="top",
        buttons=[
            dict(label="▶", method="animate", args=[None, dict(frame=dict(duration=500, redraw=True), fromcurrent=True)]),
            dict(label="⏸", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=True), mode="immediate")])
        ]
    )],
    width=120 + g * 50,
    height=240 + g * 50,
    xaxis=dict(
        range=[-0.5, g - 0.5], scaleanchor="y",
        tickmode="array", tickvals=list(range(g)),
        ticktext=[str(i) for i in range(g)], title="col",
    ),
    yaxis=dict(
        range=[g - 0.5, -0.5], autorange=False,
        tickmode="array", tickvals=list(range(g)),
        ticktext=[str(i) for i in range(g)], title="row",
    ),
    plot_bgcolor="white",
    margin=dict(l=50, r=30, t=60, b=140),
    title=f"Interactive Game Replay ({len(history)} moves, winner: {game.winner.name if game.game_over else 'ongoing'})"
)

fig.show()

In [12]:
weird_state = replay_states[-4]
weird_state.visualize()
weird_state.current_player

<Player.VERTICAL: 1>

In [13]:
mcts.get_action_probs(weird_state, temperature=0.5)

array([0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       3.5101548e-04, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       3.0266133e-04, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       2.9342031e-04, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       3.2157317e-04, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 9.9345809e-01, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       3.1204559e-04, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 3.4105810e-04, 0.0000000e+00,
       3.0266133e-04, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       5.5474776e-04, 0.0000000e+00, 0.0000000e+00, 0.0000000e

### Text-based replay (turn boundaries only)

This shows the board state only when a turn ends (when `moves_left` was 1).

## 7. Backpropagation sanity check

Verify that value signs are correct: same-player consecutive nodes should accumulate the same sign, different-player transitions flip.

In [8]:
# Walk the most-visited path from root to a leaf and check Q values
game = Bridgit(N)
root = MCTSNode(game.copy())
mcts._expand(root)

for _ in range(config.num_mcts_sims):
    node = root
    while node.is_expanded and node.children:
        node = node.best_child(config.c_puct)
    if node.game.game_over:
        mcts._backpropagate(node, 1.0)
        continue
    value = mcts._expand(node)
    mcts._backpropagate(node, value)

# Trace the most-visited path
print("Most-visited path from root:\n")
print(f"{'Depth':>5} {'Move':>8} {'Player':>12} {'MovesLeft':>10} {'Visits':>7} {'Q':>7} {'Same as parent?':>16}")
print("-" * 75)

node = root
depth = 0
print(f"{depth:>5} {'root':>8} {node.game.current_player.name:>12} {node.game.moves_left_in_turn:>10} {node.visit_count:>7} {node.q_value:>7.3f} {'':>16}")

while node.children:
    best_a = max(node.children, key=lambda a: node.children[a].visit_count)
    child = node.children[best_a]
    depth += 1
    r, c = gw.action_to_move(best_a)
    same = "YES" if child.game.current_player == node.game.current_player else "no (flip)"
    print(f"{depth:>5} {f'({r},{c})':>8} {child.game.current_player.name:>12} {child.game.moves_left_in_turn:>10} {child.visit_count:>7} {child.q_value:>7.3f} {same:>16}")
    node = child
    if depth > 8:
        print("  ...")
        break

Most-visited path from root:

Depth     Move       Player  MovesLeft  Visits       Q  Same as parent?
---------------------------------------------------------------------------
    0     root   HORIZONTAL          1    5000   0.025                 
    1    (3,7)     VERTICAL          2     209  -0.017        no (flip)
    2    (2,2)     VERTICAL          1       6  -0.027              YES
    3    (4,8)   HORIZONTAL          2       3   0.026        no (flip)
    4    (3,3)   HORIZONTAL          1       2  -0.004              YES
    5    (1,5)     VERTICAL          2       1   0.096        no (flip)
    6    (1,1)     VERTICAL          1       0   0.000              YES


In [9]:
from bridgit.ai import RandomPlayer, MCTSPlayer, GreedyMCTSPlayer, Arena

# Create different players
random_player = RandomPlayer(board_size=N, name="Random")
mcts_player = MCTSPlayer(wrapper, config, temperature=1.0, name="MCTS-T1.0")
greedy_mcts = GreedyMCTSPlayer(wrapper, config, name="MCTS-Greedy")

# Create arena
arena = Arena(mcts_player, random_player, config)

# Play a few games
num_games = 10
p1_wins, p2_wins, draws = arena.play_games(num_games, verbose=False)

print(f"\n{mcts_player.name} vs {random_player.name}")
print(f"Results after {num_games} games:")
print(f"  {mcts_player.name}: {p1_wins} wins ({p1_wins/num_games*100:.1f}%)")
print(f"  {random_player.name}: {p2_wins} wins ({p2_wins/num_games*100:.1f}%)")
print(f"\nMCTS should dominate random play even with an untrained net!")

KeyboardInterrupt: 

In [ ]:
import pandas as pd

# Create multiple players
players = [
    RandomPlayer(board_size=N, name="Random"),
    MCTSPlayer(wrapper, config, temperature=1.5, name="MCTS-T1.5"),
    MCTSPlayer(wrapper, config, temperature=1.0, name="MCTS-T1.0"),
    GreedyMCTSPlayer(wrapper, config, name="MCTS-Greedy"),
]

# Round-robin: each player plays against every other player
games_per_matchup = 4
results = []

for i, p1 in enumerate(players):
    for j, p2 in enumerate(players):
        if i >= j:  # Skip self-play and duplicate matchups
            continue
        
        arena = Arena(p1, p2, config)
        p1_wins, p2_wins, _ = arena.play_games(games_per_matchup, verbose=False)
        
        results.append({
            "Player 1": p1.name,
            "Player 2": p2.name,
            "P1 Wins": p1_wins,
            "P2 Wins": p2_wins,
            "P1 Win%": f"{p1_wins/games_per_matchup*100:.1f}%"
        })
        
        print(f"{p1.name} vs {p2.name}: {p1_wins}-{p2_wins}")

# Display results table
df = pd.DataFrame(results)
print(f"\n{'='*60}")
print("LEAGUE RESULTS")
print(f"{'='*60}")
print(df.to_string(index=False))

ModuleNotFoundError: No module named 'pandas'

### League-style tournament

Compare multiple players in a round-robin tournament.

## 8. Player abstraction — compare different strategies

Use the Player classes to pit different strategies against each other.